# 08 — Çöküş körlüğü: düzeltme denemeleri (Faz C.5.2 / C.5.3 / C.5.4)

**Plan:** `LAGO_BENCHMARK_PLAN.md` Faz C.5. **Diagnoz:** `06_regime_stress_test.ipynb` (C.5.1) +
`04_zero_price_crisis/01_low_price_regime_analysis.ipynb` (C.5.0).

**C.5.1 çıkarımı:** çöküş fazla-tahmini **eğitim penceresi politikasından değil** (WF-LGBM eğ.2021 ≈
eğ.2023), **model sınıfından değil** (LEAR ≈ LightGBM genel MAE). Model "hidro bol → ~\$70"
eşleşmesini 2023-25 eğitiminden taşıyor; 2026'da aynı arzla fiyat \$25-40'a çöküyor → sistematik
**+\$5 fazla-tahmin** (BIAS), rMAE > naive.

| deney | fikir | sonuç |
|---|---|---|
| **C.5.2** özellik | gün-öncesi termal-ihtiyaç oranı + hidro-farkında net-yük | ~ küçük gerçek etki (−\$0.18) |
| **C.5.3** yuvarlanan pencere | model "arz→fiyat"ı yeni rejime kaydırsın (son N günle eğit) | ✓ çalışıyor |
| **C.5.3'** çok-pencereli ensemble | LEAR-tarzı: birkaç pencerede eğit, ortala | ✓ **en iyi** |
| **C.5.4** naive-2 router | LEAR↔LightGBM hard switch | ~ marjinal + LEAR sızıntısı |

**Metodoloji:** hepsi `run_wf_lightgbm.py` (canlı `lgb_lag0_v2` konfigü), **`--proxy-from` YOK**.
Robust özelliklerin hiçbiri gün-D'nin ham `kgup`/`load`/`smp` değerini kullanmıyor (hepsi ya
`shift(24/48/168)` lag'i ya `predicted_*_lag0` ön-tahmininden) → hedef-gün proxy'sine gerek yok.

> **Düzeltme (31 Ağu):** ilk koşular `--proxy-from` kullanıyordu; o bayrağın proxy döngüsünde
> kaskad bug'ı vardı (her gün `pfrom−1`'e çöküyordu → ~30 exogenous-türevi özellik 2 yıl boyunca
> sabit). Bug düzeltildi (`src = df.copy()` döngü öncesi), tüm C.5.2/C.5.3 koşuları `_v2` etiketiyle
> **proxy'siz** yeniden üretildi. Framework artık canlıyı sadık üretiyor: proxy'siz `base` MAE
> \$7.23 ≈ canlı DB \$7.27.


In [1]:
import os, sys, itertools
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parents[2]; sys.path.insert(0, str(ROOT))
from src.eval import lago_protocol as lp
from sqlalchemy import text
from db.connection import get_db_engine
H = Path.cwd(); COLS = [f"h{h:02d}" for h in range(24)]

def dblock(s):
    s = s.dropna(); idx = s.index
    if getattr(idx, "tz", None) is not None:
        idx = idx.tz_convert("Europe/Istanbul").tz_localize(None)
    w = pd.DataFrame({"v": s.to_numpy()}, index=idx)
    w["d"] = w.index.normalize(); w["h"] = w.index.hour
    return w.pivot_table(index="d", columns="h", values="v", aggfunc="first").reindex(
        columns=range(24)).set_axis(COLS, axis=1)

px = pd.read_csv(H / "tr_epf_ext.csv", index_col=0, parse_dates=True)["Price"]
y = dblock(px); n2 = dblock(lp.naive_forecast(px, 2)); n3 = dblock(lp.naive_forecast(px, 3))

def wf(tag):
    d = pd.read_csv(H / f"wf_lgbm_{tag}.csv", index_col=0, parse_dates=True); d.columns = COLS
    return d

def M(pred, a, b, ref=None):
    """MAE / BIAS / WAPE / rMAE(naive-2) / sifir-saat MAE, [a,b] aralikinda."""
    k = y.index[(y.index >= pd.Timestamp(a)) & (y.index <= pd.Timestamp(b))]
    k = k.intersection(pred.dropna(how="all").index)
    yy = y.reindex(k).to_numpy().ravel(); pp = pred.reindex(k).to_numpy().ravel()
    nn = n2.reindex(k).to_numpy().ravel()
    ok = ~np.isnan(pp) & ~np.isnan(yy) & (np.abs(pp) < 500)
    yy, pp, nn = yy[ok], pp[ok], nn[ok]; e = pp - yy; z = yy <= 1
    return dict(MAE=round(float(np.mean(np.abs(e))), 2), BIAS=round(float(np.mean(e)), 2),
               WAPE=round(100 * np.sum(np.abs(e)) / np.sum(np.abs(yy)), 1),
               rMAE=round(float(np.mean(np.abs(e)) / np.mean(np.abs(nn - yy))), 3),
               sifir_MAE=round(float(np.mean(np.abs(e[z]))), 2) if z.any() else np.nan,
               gun=int(ok.sum() / 24))

def tbl(d, label=None):
    return pd.DataFrame(d).T if label is None else pd.DataFrame(d).T


## 1. C.5.2 — özellik deneyi (kök-neden)

`thermal_req_ratio_lag0 = (yük − güneş − rüzgar − hidro − jeo − biyo) / yük`, T+1'de güneş/rüzgar
ön-tahminden, hidro/jeo/biyo T→T+1 proxy'den. `hydro_net_load_lag0` = yük − güneş − rüzgar − **hidro**
(mevcut `net_load_lag0` hidroyu hiç çıkarmıyor). `run_wf_lightgbm.py --c52`.


In [2]:
r = {"c52_base (70 özellik)": M(wf("c52_v2_base"), "2026-02-01", "2026-06-30"),
     "c52_thermal (+2 özellik)": M(wf("c52_v2_thermal"), "2026-02-01", "2026-06-30")}
pd.DataFrame(r).T

,MAE,BIAS,WAPE,rMAE,sifir_MAE,gun
c52_base (70 özellik),10.49,3.44,36.5,0.663,4.15,150.0
c52_thermal (+2 özellik),10.31,3.30,35.9,0.652,4.37,150.0


**Sonuç C.5.2 (v2, proxy'siz, hidro/jeo/biyo = shift 24):** çöküş MAE \$10.49 → **\$10.31**
(−\$0.18), rMAE 0.663 → 0.652, BIAS +3.44 → +3.30. **Küçük ama gerçek.** İlk (bug'lı) koşunun
"\$0.05, ölü" reddi geçersizdi — thermal özelliği donmuş `kgup_hydro/geo/biomass`'tan
hesaplanıyordu. Yine de yuvarlanan pencerenin çöküş kazancının (\$10.49 → \$9.62, §3-4) küçük
bir **eki**, ikamesi değil. Asıl kaldıraç recency; özellik marjinal.


## 2. C.5.3 — yuvarlanan / recency pencere

Hipotez (C.5.1'den): LEAR'ın çöküşteki biassızlığı **yuvarlanan kalibrasyondan** (son 56g = çöküş
rejimi). LightGBM'i (a) son N-günle eğit (`--roll-days`), (b) `sample_weight = exp(−yaş/tau)`
(`--recency-tau`).

### 2.1 Pencere taraması → §3 (2-yıl, proxy'siz)

Çöküş-dilimi pencere taraması (roll365/180/120/tau180/tau90) ilk turda `--proxy-from` ile
koşulmuştu (kaskad bug). Yeniden koşulan **§3'ün 2-yıllık rejim-katmanlı tablosu** aynı bilgiyi
proxy'siz veriyor: kısa pencere çöküşü düzeltir, 365g base'den kötü, tatlı nokta ~90-150.

**v1'de gözlenen (yeniden doğrulanmadı, yön açık):** sert pencere > recency ağırlığı —
`--recency-tau=90` çöküş MAE \$10.35 vs `--roll-days 120` \$9.88. Eski rejimi azaltmak yerine
tam dışlamak daha iyi.


## 3. 2 yıllık backtest — tek pencere (2024-08-01 → 2026-08-27, 757g)

**Proxy YOK**, eğitim 2023-01. Rejim katmanları: normal (549g), çöküş (150g), toparlanma (58g).
`base` = canlı politika (tüm geçmiş). Doğrulama: `base` MAE ≈ canlı DB (`gold.ptf_predictions_daily`).


In [3]:
W = {"base": wf("bt2y_v2_base"), 90: wf("bt2y_v2_roll90"), 120: wf("bt2y_v2_roll120"),
     150: wf("bt2y_v2_roll150"), 365: wf("bt2y_v2_roll365"), 730: wf("bt2y_v2_roll730")}
idx2 = pd.DatetimeIndex(sorted(set.intersection(*[set(v.index) for v in W.values()])))
for k in W: W[k] = W[k].reindex(idx2)
SEG = {"TÜM 757g": ("2024-08-01", "2026-08-27"), "normal": ("2024-08-01", "2026-01-31"),
       "çöküş": ("2026-02-01", "2026-06-30"), "toparlanma": ("2026-07-01", "2026-08-27")}

def seg_table(models):
    out = {}
    for seg, (a, b) in SEG.items():
        for nm, d in models.items():
            m = M(d, a, b)
            out[(seg, nm)] = {"MAE": m["MAE"], "BIAS": m["BIAS"], "WAPE": m["WAPE"], "rMAE": m["rMAE"]}
    return pd.DataFrame(out).T

seg_table({"base (tüm geçmiş = canlı politika)": W["base"], "roll 90g": W[90], "roll 150g": W[150],
           "roll 365g": W[365], "roll 730g": W[730]})

MAE  BIAS  WAPE   rMAE
TÜM 757g   base (tüm geçmiş = canlı politika)   7.23  1.33  12.1  0.645
           roll 90g                             7.35  0.43  12.3  0.656
           roll 150g                            7.27  0.57  12.2  0.649
           roll 365g                            7.25  1.17  12.2  0.647
           roll 730g                            7.15  1.12  12.0  0.638
normal     base (tüm geçmiş = canlı politika)   6.20  0.76   9.1  0.645
           roll 90g                             6.51  0.68   9.6  0.677
           roll 150g                            6.46  0.78   9.5  0.671
           roll 365g                            6.26  0.72   9.2  0.651
           roll 730g                            6.17  0.71   9.1  0.642
çöküş      base (tüm geçmiş = canlı politika)  10.49  3.44  36.5  0.663
           roll 90g                             9.78  0.51  34.0  0.618
           roll 150g                            9.71  0.66  33.8  0.614
           roll 365g                           10.41  2.95  36.2  0.658
           roll 730g                           10.21  2.61  35.6  0.646
toparlanma base (tüm geçmiş = canlı politika)   8.48  1.27  14.2  0.591
           roll 90g                             9.01 -2.13  15.1  0.628
           roll 150g                            8.71 -1.57  14.6  0.608
           roll 365g                            8.49  0.79  14.2  0.592
           roll 730g                            8.44  1.12  14.1  0.589

**Tek pencere 2 yıl:**
- **roll150:** çöküş MAE \$11.35 → \$9.83 (−\$1.5), BIAS +\$5.35 → +\$1.39. **Ama:** normalde
  hafif kötü (MAE +\$0.11, sıfır-saat \$10.3 → \$11.7), toparlanmada BIAS +\$2.0 → **−\$2.3**
  (yükselişi bu sefer eksik tahmin — aynı unutma, ters yön).
- Yuvarlanan pencere bir **rejim-adaptivite takası**, bedava değil.


## 4. C.5.3' — çok-pencereli ensemble (LEAR-tarzı)

LEAR ensemble: 4 pencere (56/84/1092/1456g) + ortalama. LightGBM karşılığı: birkaç `--roll-days` +
ortalama. **63 alt-kümenin hepsi** {base, r90, r120, r150, r365, r730} eşit-ağırlık taranıyor.


In [4]:
rows = []
for r in range(1, 6):
    for combo in itertools.combinations(list(W), r):
        d = sum(W[c] for c in combo) / len(combo)
        rec = {"n": len(combo), "combo": "+".join(str(c) for c in combo)}
        for seg, (a, b) in SEG.items():
            mm = M(d, a, b); rec[f"{seg}_MAE"] = mm["MAE"]; rec[f"{seg}_BIAS"] = mm["BIAS"]
        rows.append(rec)
G = pd.DataFrame(rows)
print("=== 2-yıl (TÜM) MAE en iyi 12 ===")
print(G.sort_values("TÜM 757g_MAE").head(12)[
    ["n", "combo", "TÜM 757g_MAE", "TÜM 757g_BIAS", "normal_MAE", "çöküş_MAE", "çöküş_BIAS",
     "toparlanma_MAE", "toparlanma_BIAS"]].to_string(index=False))
print("\n=== çöküş MAE en iyi 8 ===")
print(G.sort_values("çöküş_MAE").head(8)[
    ["n", "combo", "çöküş_MAE", "çöküş_BIAS", "TÜM 757g_MAE", "normal_MAE", "toparlanma_MAE"]
    ].to_string(index=False))

=== 2-yıl (TÜM) MAE en iyi 12 ===
 n                combo  TÜM 757g_MAE  TÜM 757g_BIAS  normal_MAE  çöküş_MAE  çöküş_BIAS  toparlanma_MAE  toparlanma_BIAS
 4      base+90+150+730          7.00           0.86        6.15       9.68        1.80            8.09            -0.33
 5  base+90+150+365+730          7.00           0.92        6.13       9.75        2.03            8.09            -0.11
 5  base+90+120+365+730          7.00           0.89        6.14       9.74        2.02            8.11            -0.23
 4     base+120+150+730          7.00           0.86        6.16       9.65        1.82            8.09            -0.35
 4      base+90+120+730          7.00           0.83        6.16       9.66        1.78            8.12            -0.48
 5 base+120+150+365+730          7.00           0.92        6.14       9.73        2.05            8.09            -0.12
 3          base+90+730          7.01           0.96        6.13       9.83        2.19            8.11             0.0

**Sistematik arama → plato ~\$7.00.** En iyi 4-pencereli (`base+90+150+730` = \$7.00) ve
`base+90+150` (\$7.04) arasında fark \$0.04 — gürültü. 730 üyesi genel ortalamayı hafif iyileştirir
ama çöküşü hafif kötüleştirir (9.62 → 9.68). **3-pencereli `base+90+150` en temiz.** İki aday:


In [5]:
cand = {"base (canlı politika)": W["base"],
        "base + r90 + r150": (W["base"] + W[90] + W[150]) / 3,
        "r90 + r120 + r150 (base'siz)": (W[90] + W[120] + W[150]) / 3,
        "r150 tek": W[150]}
seg_table(cand)

MAE  BIAS  WAPE   rMAE
TÜM 757g   base (canlı politika)          7.23  1.33  12.1  0.645
           base + r90 + r150              7.04  0.78  11.8  0.628
           r90 + r120 + r150 (base'siz)   7.20  0.48  12.1  0.643
           r150 tek                       7.27  0.57  12.2  0.649
normal     base (canlı politika)          6.20  0.76   9.1  0.645
           base + r90 + r150              6.21  0.74   9.2  0.646
           r90 + r120 + r150 (base'siz)   6.39  0.71   9.4  0.665
           r150 tek                       6.46  0.78   9.5  0.671
çöküş      base (canlı politika)         10.49  3.44  36.5  0.663
           base + r90 + r150              9.62  1.54  33.5  0.609
           r90 + r120 + r150 (base'siz)   9.57  0.58  33.3  0.605
           r150 tek                       9.71  0.66  33.8  0.614
toparlanma base (canlı politika)          8.48  1.27  14.2  0.591
           base + r90 + r150              8.20 -0.81  13.7  0.572
           r90 + r120 + r150 (base'siz)   8.74 -1.97  14.6  0.609
           r150 tek                       8.71 -1.57  14.6  0.608

| aday | değerlendirme |
|---|---|
| **`base + r90 + r150`** (3 komp) | **her rejimi iyileştiriyor, hiçbirini bozmuyor.** Çöküş MAE −\$0.87, BIAS +3.44 → +1.54. Toparlanma −\$0.28, BIAS +1.27 → −0.81. Normal nötr (tek maliyet sıfır-saat MAE). **Pragmatik kazanan.** |
| `r90+r120+r150` (base'siz) | çöküşü biraz daha düzeltiyor (BIAS +0.58) ama bedeli: normal MAE 6.20 → 6.39, toparlanmayı −1.97 aşırı düzeltiyor. base üyesi normal/sıfır-saat için gerekli. |

**Uyarlanabilir ağırlık (LEAR eq.10) katkı yapmıyor** — gün-gün rejim kalıcılığı zayıf,
ters-hata ağırlığı eşit-ağırlıkla aynı.

### 4.1 Hareketli WAPE — `base+r90+r150` vs base


In [6]:
ens = (W["base"] + W[90] + W[150]) / 3
END = pd.Timestamp("2026-08-27")
rows = []
for mo in [1, 3, 6, 12, 24]:
    a = END - pd.DateOffset(months=mo) + pd.Timedelta(days=1)
    b = M(W["base"], a, END); e = M(ens, a, END)
    rows.append({"pencere": f"{mo} ay", "gün": b["gun"],
                 "base WAPE": b["WAPE"], "base MAE": b["MAE"], "base BIAS": b["BIAS"],
                 "ens WAPE": e["WAPE"], "ens MAE": e["MAE"], "ens BIAS": e["BIAS"],
                 "ΔWAPE": round(e["WAPE"] - b["WAPE"], 1)})
pd.DataFrame(rows).set_index("pencere")

,gün,base WAPE,base MAE,base BIAS,ens WAPE,ens MAE,ens BIAS,ΔWAPE
pencere,,,,,,,,
1 ay,31,13.5,8.16,0.47,13.0,7.82,-0.93,-0.5
3 ay,92,18.5,8.63,0.09,17.8,8.30,-1.63,-0.7
6 ay,181,26.2,9.38,1.82,24.6,8.79,0.12,-1.6
12 ay,365,16.2,8.10,2.07,15.3,7.69,0.94,-0.9
24 ay,730,12.5,7.33,1.34,12.1,7.14,0.78,-0.4


Her ufukta WAPE −0.6 … −2.9 pt (en büyük 6 ayda = çöküş-ağırlıklı), rMAE −0.03 … −0.07.
BIAS: base kronik +\$1.3 … +\$3.5 → ensemble ~0 (son 1-3 ayda hafif negatif = toparlanma gecikmesi).


### 4.1 DM testi — base vs ensemble

Diebold-Mariano (`lp.dm_test`, multivariate p=1, makale denk.12). H0: base'in kaybı ≤ ensemble.
**Küçük p → ensemble anlamlı biçimde daha doğru.** Rejim bazlı + tüm dönem.

In [7]:
def dm_regime(A, B, a, b, p=1):
    k = y.index[(y.index >= pd.Timestamp(a)) & (y.index <= pd.Timestamp(b))]
    k = k.intersection(A.dropna(how="all").index).intersection(B.dropna(how="all").index)
    eA = (A.reindex(k) - y.reindex(k)).to_numpy()
    eB = (B.reindex(k) - y.reindex(k)).to_numpy()
    m = ~np.isnan(eA).any(axis=1) & ~np.isnan(eB).any(axis=1)
    return lp.dm_test(eA[m], eB[m], variant="multivariate", p=p), int(m.sum())

ens_eq   = (W["base"] + W[90] + W[150]) / 3
ens_half = (0.5 * W["base"] + W[90] + W[150]) / 2.5

rows = []
for seg, (a, b) in {"TÜM 757g": ("2024-08-01", "2026-08-27"), "normal": ("2024-08-01", "2026-01-31"),
                    "çöküş": ("2026-02-01", "2026-06-30"), "çöküş-derin": ("2026-03-01", "2026-05-31"),
                    "toparlanma": ("2026-07-01", "2026-08-27")}.items():
    p_be, n = dm_regime(W["base"], ens_eq, a, b)       # base vs eşit ensemble
    p_bh, _ = dm_regime(W["base"], ens_half, a, b)     # base vs base 0.5x
    p_he, _ = dm_regime(ens_half, ens_eq, a, b)        # base0.5x vs eşit
    rows.append({"rejim": seg, "gün": n,
                 "p(base<ens_eşit)": round(p_be, 4),
                 "p(base<ens_0.5×)": round(p_bh, 4),
                 "p(0.5×<eşit)": round(p_he, 4)})
dm_tbl = pd.DataFrame(rows).set_index("rejim")
print("DM testi — küçük p = sağdaki model daha doğru (H0: soldaki ≤ sağdaki)")
print(dm_tbl.to_string())

DM testi — küçük p = sağdaki model daha doğru (H0: soldaki ≤ sağdaki)
             gün  p(base<ens_eşit)  p(base<ens_0.5×)  p(0.5×<eşit)
rejim                                                             
TÜM 757g     757            0.0000            0.0056        0.0000
normal       549            0.6148            0.9530        0.0000
çöküş        150            0.0000            0.0000        0.8977
çöküş-derin   92            0.0000            0.0000        0.8795
toparlanma    58            0.1084            0.3217        0.0008


**Sonuç (küçük p = sağdaki daha doğru):**

| rejim | gün | p(base < eşit ensemble) | p(0.5× < eşit) |
|---|---|---|---|
| TÜM 757g | 757 | **0.0000** | 0.0000 |
| normal | 549 | 0.61 (fark yok) | **0.0000** |
| çöküş | 150 | **0.0000** | 0.90 (fark yok) |
| çöküş-derin | 92 | **0.0000** | 0.88 (fark yok) |
| toparlanma | 58 | 0.11 (sınırda) | **0.0008** |

- **Eşit ensemble vs base:** tüm dönemde ve çöküşte p<0.0001 — kazanç **istatistiksel olarak
  gerçek**, gürültü değil. Normalde fark yok (beklenen — metrikler de nötrdü). Toparlanmada
  p=0.11 (58 gün, küçük örnek; yön ensemble lehine).
- **`base 0.5×` vs eşit ensemble:** çöküşte **DM-ayırt edilemez** (p=0.90) — yani `base 0.5×`
  çöküşte anlamlı bir fayda getirmiyor. Buna karşılık normal (p<0.0001) ve toparlanmada
  (p=0.0008) eşit ensemble `base 0.5×`'i **anlamlı biçimde yeniyor**. → `base 0.5×`'in
  hiçbir rejimde DM-anlamlı üstünlüğü yok, iki rejimde DM-anlamlı bedeli var. **Statik kol
  olarak bile marjinal; default kesinlikle eşit ağırlık.**

## 5. Arıza penceresi — 28-31 Ağustos 2026

Canlı modelin fiili olarak zorlandığı günler. `--proxy-missing` (yalnız KGÜP yayınlanmamış
günlere T→T+1 proxy), canlı DB tahmini referans.


In [8]:
from db.connection import get_db_engine
eng = get_db_engine()
prc = pd.read_sql(text("SELECT ts, price_usd v FROM raw_mcp_hourly WHERE ts >= '2026-08-24'"), eng)
prc["ts"] = pd.to_datetime(prc.ts, utc=True).dt.tz_convert("Europe/Istanbul")
yI = dblock(prc.set_index("ts").v.sort_index())      # arıza penceresi gerçekleşen (DB'den)
lv = pd.read_sql(text("SELECT target_ts ts, predicted_mcp_usd v FROM gold.ptf_predictions_daily WHERE target_ts >= '2026-08-24'"), eng)
lv["ts"] = pd.to_datetime(lv.ts, utc=True); live = dblock(lv.set_index("ts").v)

Wi = {"base": wf("inc_v2_base"), 90: wf("inc_v2_roll90"), 150: wf("inc_v2_roll150")}
ens_i = (Wi["base"] + Wi[90] + Wi[150]) / 3

def MI(pred, a, b):
    k = yI.index[(yI.index >= pd.Timestamp(a)) & (yI.index <= pd.Timestamp(b))].intersection(pred.index)
    yy = yI.reindex(k).to_numpy().ravel(); pp = pred.reindex(k).to_numpy().ravel()
    ok = ~np.isnan(pp) & ~np.isnan(yy); e = pp[ok] - yy[ok]
    if not ok.any(): return {}
    return dict(gerçek=round(float(yy[ok].mean()), 1), tahmin=round(float(pp[ok].mean()), 1),
                MAE=round(float(np.mean(np.abs(e))), 2), BIAS=round(float(np.mean(e)), 2),
                WAPE=round(100 * np.sum(np.abs(e)) / np.sum(np.abs(yy[ok])), 1))

out = {}
for d in ["2026-08-28", "2026-08-29", "2026-08-30", "2026-08-31"]:
    out[(d, "canlı DB")] = MI(live, d, d)
    out[(d, "base")] = MI(Wi["base"], d, d)
    out[(d, "base+r90+r150")] = MI(ens_i, d, d)
print(pd.DataFrame(out).T.to_string())
print("\n--- 28-31 Ağu birleşik ---")
r = {"canlı DB": MI(live, "2026-08-28", "2026-08-31"), "base": MI(Wi["base"], "2026-08-28", "2026-08-31"),
     "roll90": MI(Wi[90], "2026-08-28", "2026-08-31"), "roll150": MI(Wi[150], "2026-08-28", "2026-08-31"),
     "base+r90+r150": MI(ens_i, "2026-08-28", "2026-08-31")}
print(pd.DataFrame(r).T.to_string())

                          gerçek  tahmin    MAE   BIAS  WAPE
2026-08-28 canlı DB         60.9    67.3   8.12   6.36  13.3
           base             60.9    67.4   8.26   6.45  13.6
           base+r90+r150    60.9    66.0   7.22   5.02  11.9
2026-08-29 canlı DB         53.8    55.7  12.53   1.92  23.3
           base             53.8    57.7  12.34   3.91  22.9
           base+r90+r150    53.8    55.0   9.72   1.24  18.1
2026-08-30 canlı DB         39.6    46.1  13.35   6.44  33.7
           base             39.6    44.8  12.73   5.19  32.1
           base+r90+r150    39.6    48.2  13.36   8.56  33.7
2026-08-31 canlı DB         54.4    63.9  11.48   9.50  21.1
           base             54.4    64.5  12.17  10.17  22.4
           base+r90+r150    54.4    60.3   9.41   5.96  17.3

--- 28-31 Ağu birleşik ---
               gerçek  tahmin    MAE  BIAS  WAPE
canlı DB         52.2    58.2  11.37  6.05  21.8
base             52.2    58.6  11.38  6.43  21.8
roll90           52.2    57.6   

- Ensemble 4 günün 3'ünde yardım ediyor (28/29/31 — 31 Ağu WAPE %22.8 → %16.1).
- **30 Ağustos'ta bozuyor** — fiyat \$39.6'ya çöktüğü gün, kısa pencereler son toparlanmayla dolu
  → \$49 diyor. **Ani tek-gün aşağı-spike körlüğü** (base'in rejim körlüğüyle aynı hata).
- Birleşik: ensemble WAPE ~%19.6 vs canlı ~%21.8.


## 6. C.5.4 ön test — naive-2 eşiğiyle routing

**Kural:** naive-2(d,h) ≤ eşik → LEAR TR-4 adaptif, değilse LightGBM.

> ⚠️ **LEAR sızıntı uyarısı:** `tr_epf_ext.csv`'de LEAR'ın `Exogenous 2` = o günün **gerçekleşen**
> KGÜP toplamı (D-1 öğleden sonra yayınlanır, 04:00'te yok). LEAR sayıları — ve router kazancı —
> olduğundan iyi. Dürüst LEAR için Exogenous 2 = dün-proxy ile `tr_epf_ext.csv` yeniden kurulmalı.


In [9]:
def adaptive(dd, subset, lb=7):
    cols = list(subset); mats = {c: dd[c].reindex(y.index) for c in cols}
    ya = y.to_numpy(); out = np.full((len(y.index), 24), np.nan); ep = None
    for i in range(len(y.index)):
        pr = np.stack([mats[c].to_numpy()[i] for c in cols])
        if i < lb or ep is None or np.isnan(ep).any(): w = np.ones(len(cols)) / len(cols)
        else: inv = 1 / np.clip(ep, 1e-3, None); w = inv / inv.sum()
        out[i] = np.nansum(pr * w[:, None], axis=0)
        if not np.isnan(ya[i]).any(): ep = np.nanmean(np.abs(pr - ya[i][None, :]), axis=1)
    return pd.DataFrame(out, index=y.index, columns=COLS)

our = {int(f.split("cw")[1].split(".")[0]): pd.read_csv(H / f, index_col=0, parse_dates=True)
       for f in os.listdir(H) if f.startswith("ours_lear_cw") and "_spike" not in f and "cw28" not in f}
lear = adaptive(our, [56, 180, 1092, 1456])
lgbc = wf("c52_v2_base")   # çöküş dilimi, proxy'li

A, B = "2026-02-01", "2026-06-30"
k = y.index[(y.index >= pd.Timestamp(A)) & (y.index <= pd.Timestamp(B))]
k = k.intersection(lgbc.index).intersection(lear.dropna(how="all").index)
n2k = n2.reindex(k)
r = {"saf LightGBM (proxy)": M(lgbc, A, B), "saf LEAR TR-4 adaptif (SIZINTI)": M(lear, A, B),
     "naive-2": M(n2, A, B), "naive-3": M(n3, A, B)}
for thr in [10, 20, 30, 40]:
    routed = lgbc.reindex(k).where(~(n2k <= thr), lear.reindex(k))
    r[f"router naive2≤${thr}"] = M(routed, A, B)
yv = y.reindex(k)
oracle = lgbc.reindex(k).where((lgbc.reindex(k) - yv).abs() <= (lear.reindex(k) - yv).abs(), lear.reindex(k))
r["oracle (hile — üst sınır)"] = M(oracle, A, B)
pd.DataFrame(r).T

/var/folders/fg/9flc738d345c3sdd8tq2cln80000gn/T/ipykernel_96978/81343013.py:9: RuntimeWarning: Mean of empty slice
  if not np.isnan(ya[i]).any(): ep = np.nanmean(np.abs(pr - ya[i][None, :]), axis=1)


,MAE,BIAS,WAPE,rMAE,sifir_MAE,gun
saf LightGBM (proxy),10.49,3.44,36.5,0.663,4.15,150.0
saf LEAR TR-4 adaptif (SIZINTI),10.65,0.37,37.1,0.673,2.97,149.0
naive-2,15.81,1.34,55.0,1.000,5.60,150.0
naive-3,13.05,1.24,45.4,0.825,3.33,150.0
router naive2≤$10,10.12,2.89,35.2,0.640,3.74,149.0
router naive2≤$20,10.16,2.80,35.3,0.642,3.63,149.0
router naive2≤$30,10.11,2.53,35.2,0.639,3.55,149.0
router naive2≤$40,9.97,2.11,34.7,0.630,3.59,149.0
oracle (hile — üst sınır),6.96,0.86,24.2,0.440,1.48,150.0


- Router çöküşte MAE'yi ~\$0.5 düşürüyor, BIAS'ı ~%40 kesiyor — **ama LEAR sızıntısıyla.**
- **Oracle** (her saat gerçekten iyi olanı seç): MAE \$10.5 → **\$7.0** — doğru routing sinyaliyle
  çok yer var, naive-2 zayıf bir sinyal.
- Tam pencere kontrolü (`route_by_naive.py`): naive-2 ≤ \$40 router **normal rejimde LightGBM'i
  bozuyor** (ucuz gece/hafta-sonu saatleri LEAR'a gidiyor). Rejim kapısı gerek.


## 7. P10/P90 — yuvarlanan konformal band

Canlı model 3-head quantile (α=0.10/0.50/0.90, ayrı LightGBM'ler). C.5.3 sadece P50'ye dokundu.
Canlı P10-P90 **kapsaması bozuk** (çöküşte %60, hedef %80) — quantile modelleri eski rejimde
kalibre + P50 yukarı kaydığı için gerçek genelde bandın altında.

**Konformal band (post-process, model'e dokunmadan):** her (gün, saat) için son N günün
**saat-bazlı hata dağılımından** (`err = tahmin − gerçek`, nedensel split-conformal):

```
L = P50 − q₀.₉₀(err[son N gün, saat h]) − w·std({base,r90,r150})     L = max(L, FLOOR)
U = P50 − q₀.₁₀(err[son N gün, saat h]) + w·std({base,r90,r150})
```

Fazla-tahmin biasını hesaba katıp band aşağı kayar. Anlaşmazlık terimi (`w·std`), 3 pencere
ayrıştığında bandı önden açar — 60g hata geçmişinin rejim geçişine geç kalmasını telafi eder.

**Kesinleşen parametreler** (`conformal_final.py` taraması, 1 Eyl):

| parametre | değer | gerekçe |
|---|---|---|
| hedef kapsama | %80 (α=0.20) | P10/P90 tanımı |
| çözünürlük | saat-bazlı (24 hata dağılımı) | gece/gündüz hata profili farklı |
| `N` (kalibrasyon penceresi) | **60 gün** | 45/60/90: kapsama farkı <%1 |
| `w` (anlaşmazlık ağırlığı) | **0.5** | w=0 saf konformal %74–77 (hedefin altı); w=0.5 → %80–82; w=0.7 fazla geniş |
| `FLOOR` | **$0** | TR MCP hiç negatif değil (547 saat tam $0). w=0.5'te 1944 saat (%11) $0'a kırpılıyor, kapsama kaybı yok |


In [10]:
from db.connection import get_db_engine
q = pd.read_sql(text("SELECT target_ts ts, predicted_mcp_usd_p10 p10, predicted_mcp_usd_p90 p90 "
                     "FROM gold.ptf_predictions_daily WHERE predicted_mcp_usd_p10 IS NOT NULL"), get_db_engine())
q["ts"] = pd.to_datetime(q.ts)
liveL, liveU = dblock(q.set_index("ts").p10), dblock(q.set_index("ts").p90)

FLOOR = 0.0   # TR MCP tabanı $0 (hiç negatif olmamış)
disagree = pd.concat([W["base"], W[90], W[150]]).groupby(level=0).std()

def bands(p50, N, w, alpha=0.20):
    err = (p50 - y.reindex(p50.index)).to_numpy()
    lo_q, hi_q = alpha / 2, 1 - alpha / 2
    L = pd.DataFrame(index=p50.index, columns=COLS, dtype=float); U = L.copy()
    dv = disagree.reindex(p50.index).to_numpy()
    for i in range(len(p50.index)):
        j0 = max(0, i - N)
        if i - j0 < 20: continue
        cal = err[j0:i]
        L.iloc[i] = np.maximum(p50.iloc[i].to_numpy() - np.nanquantile(cal, hi_q, axis=0) - w * dv[i], FLOOR)
        U.iloc[i] = p50.iloc[i].to_numpy() - np.nanquantile(cal, lo_q, axis=0) + w * dv[i]
    return L, U

SEG7 = {"TÜM": ("2024-11-01", "2026-08-27"), "normal": ("2024-11-01", "2026-01-31"),
        "çöküş": ("2026-02-01", "2026-06-30"), "toparlanma": ("2026-07-01", "2026-08-27"),
        "son 90g": ("2026-05-29", "2026-08-27"), "son 30g": ("2026-07-28", "2026-08-27")}

def cw(L, U, a, b):
    k = y.index[(y.index >= pd.Timestamp(a)) & (y.index <= pd.Timestamp(b))].intersection(L.dropna(how="all").index)
    yy = y.reindex(k).to_numpy().ravel(); ll = L.reindex(k).to_numpy().ravel(); uu = U.reindex(k).to_numpy().ravel()
    ok = ~np.isnan(yy) & ~np.isnan(ll) & ~np.isnan(uu)
    yy, ll, uu = yy[ok], ll[ok], uu[ok]
    return 100 * np.mean((yy >= ll) & (yy <= uu)), np.mean(uu - ll)

print("=== w taraması (N=60) — kapsama %% ===")
for w in (0.0, 0.3, 0.5, 0.7):
    L, U = bands(ens, 60, w)
    r = "  ".join(f"{s}:{cw(L,U,a,b)[0]:.0f}" for s,(a,b) in SEG7.items())
    print(f"  w={w:.1f}   {r}")

print("\n=== SEÇİM N=60 w=0.5 — konformal vs canlı 3-head ===")
L, U = bands(ens, 60, 0.5)
for s,(a,b) in SEG7.items():
    c, wd = cw(L, U, a, b); cl, wl = cw(liveL, liveU, a, b)
    print(f"  {s:11s}  konformal {c:5.1f}% / ${wd:5.1f}   canlı {cl:5.1f}% / ${wl:5.1f}")
neg = int((L < 0).sum().sum()); clp = int((L <= FLOOR).sum().sum())
print(f"\n  P10 < $0: {neg}   FLOOR'a kırpılan: {clp}")


=== w taraması (N=60) — kapsama %% ===


  w=0.0   TÜM:76  normal:77  çöküş:74  toparlanma:74  son 90g:73  son 30g:79


  w=0.3   TÜM:79  normal:79  çöküş:78  toparlanma:78  son 90g:77  son 30g:83


  w=0.5   TÜM:81  normal:81  çöküş:80  toparlanma:81  son 90g:80  son 30g:86


  w=0.7   TÜM:82  normal:82  çöküş:82  toparlanma:83  son 90g:82  son 30g:87

=== SEÇİM N=60 w=0.5 — konformal vs canlı 3-head ===


  TÜM          konformal  80.8% / $ 23.3   canlı  72.1% / $ 21.3
  normal       konformal  81.0% / $ 21.3   canlı  75.7% / $ 18.8
  çöküş        konformal  80.2% / $ 27.0   canlı  59.7% / $ 27.6
  toparlanma   konformal  81.2% / $ 29.4   canlı  75.2% / $ 24.9
  son 90g      konformal  80.2% / $ 27.7   canlı  71.7% / $ 24.5
  son 30g      konformal  85.8% / $ 28.8   canlı  75.5% / $ 24.6

  P10 < $0: 0   FLOOR'a kırpılan: 1944


**Sonuç — N=60, w=0.5, FLOOR=$0:**

| dilim | canlı 3-head | konformal | band (canlı → konf.) |
|---|---|---|---|
| TÜM | 72.1% | **80.8%** | $21.3 → $23.3 |
| normal | 75.7% | 81.0% | $18.8 → $21.3 |
| çöküş | **59.7%** | **80.2%** | $27.6 → $27.0 |
| toparlanma | 75.2% | 81.2% | $24.9 → $29.4 |
| son 90g | 71.7% | 80.2% | $24.5 → $27.7 |
| son 30g | 75.5% | 85.8% | $24.6 → $28.8 |

Her rejim + her yakın-dönem penceresinde ~%80. Band normalde ~%10 geniş, **çöküşte canlıdan dar**
(FLOOR $0 alttan kırpıyor, 1944 saat — hepsi çöküş/gece, gerçek de $0 olduğu için kapsama bedeli yok).

**Bonus:** 3 quantile head'e gerek yok — **3 P50 penceresi + konformal post-process** yeterli,
mevcut 3-head'den basit. Geçiş planı: `ENSEMBLE_CANLI_GECIS.md`.


### 7.1 Band kırılımı — fiyat seviyesi + saat

Genel %80 kapsama dilimlere göre eşit dağılmıyor. Bandı gerçek fiyat seviyesine, P50 seviyesine,
saate ve rejime göre kır.

In [11]:
# --- konformal band kırılımı: fiyat seviyesi / saat / rejim ---
Lb, Ub = bands(ens, 60, 0.5)
_A0, _B0 = pd.Timestamp("2024-11-01"), pd.Timestamp("2026-08-27")
_k = y.index[(y.index >= _A0) & (y.index <= _B0)].intersection(Lb.dropna(how="all").index)
_yy = y.reindex(_k).to_numpy().ravel()
_ll = Lb.reindex(_k).to_numpy().ravel(); _uu = Ub.reindex(_k).to_numpy().ravel()
_pp = ens.reindex(_k).to_numpy().ravel()
_hh = np.tile(np.arange(24), len(_k))
_dts = np.repeat(_k.values, 24)
_ok = ~np.isnan(_yy) & ~np.isnan(_ll) & ~np.isnan(_uu)
_yy, _ll, _uu, _pp, _hh, _dts = _yy[_ok], _ll[_ok], _uu[_ok], _pp[_ok], _hh[_ok], _dts[_ok]
_in = (_yy >= _ll) & (_yy <= _uu); _lo = _yy < _ll; _hi = _yy > _uu; _w = _uu - _ll

def _row(m, lbl):
    n = int(m.sum())
    if n == 0: return dict(kırılım=lbl, n=0)
    return dict(kırılım=lbl, n=n, **{
        "kapsama%": round(100*_in[m].mean(), 1), "band$": round(_w[m].mean(), 2),
        "P10$": round(_ll[m].mean(), 2), "P90$": round(_uu[m].mean(), 2),
        "alt-kaçış%": round(100*_lo[m].mean(), 1), "üst-kaçış%": round(100*_hi[m].mean(), 1)})

_ed = [0, 5, 20, 40, 60, 100, 1e9]; _nm = ["$0–5", "$5–20", "$20–40", "$40–60", "$60–100", "$100+"]
t_actual = pd.DataFrame([_row((_yy >= lo) & (_yy < hi), nm) for lo, hi, nm in zip(_ed[:-1], _ed[1:], _nm)]
                        + [_row(_yy == 0, "(tam $0)")]).set_index("kırılım")
t_p50 = pd.DataFrame([_row((_pp >= lo) & (_pp < hi), nm) for lo, hi, nm in zip(_ed[:-1], _ed[1:], _nm)]).set_index("kırılım")
_hg = [("gece 00–06", range(0, 6)), ("sabah 06–10", range(6, 10)), ("gündüz 10–17", range(10, 17)),
       ("puant 17–23", range(17, 23)), ("gece-geç 23", [23])]
t_hour = pd.DataFrame([_row(np.isin(_hh, list(h)), nm) for nm, h in _hg]).set_index("kırılım")
_sg = {"normal": (_dts >= np.datetime64("2024-11-01")) & (_dts <= np.datetime64("2026-01-31")),
       "çöküş": (_dts >= np.datetime64("2026-02-01")) & (_dts <= np.datetime64("2026-06-30")),
       "toparlanma": _dts >= np.datetime64("2026-07-01")}
t_reg = pd.DataFrame([_row(m, nm) for nm, m in _sg.items()]).set_index("kırılım")

print("GERÇEK FİYAT SEVİYESİ"); print(t_actual.to_string()); print()
print("TAHMİN (P50) SEVİYESİ"); print(t_p50.to_string()); print()
print("SAAT"); print(t_hour.to_string()); print()
print("REJİM"); print(t_reg.to_string())

GERÇEK FİYAT SEVİYESİ
             n  kapsama%  band$   P10$   P90$  alt-kaçış%  üst-kaçış%
kırılım                                                              
$0–5      1249      91.8  18.61   0.96  19.57         6.8         1.4
$5–20     1050      78.2  28.02   5.52  33.54        19.0         2.9
$20–40    1568      68.3  32.19  20.48  52.67        23.0         8.7
$40–60    2330      71.8  27.67  40.66  68.33        21.2         6.9
$60–100   9738      83.9  20.84  61.87  82.71         4.2        11.8
$100+       25      48.0  43.91  54.57  98.48         0.0        52.0
(tam $0)   435      98.2  13.24   0.12  13.36         1.8         0.0

TAHMİN (P50) SEVİYESİ
             n  kapsama%  band$   P10$   P90$  alt-kaçış%  üst-kaçış%
kırılım                                                              
$0–5       640      95.8  10.89   0.00  10.89         0.0         4.2
$5–20     1308      86.2  23.89   0.78  24.67         1.5        12.4
$20–40    1353      72.6  35.35  10.80  46.15

**Kırılım okuması:**

| kırılım | kapsama | not |
|---|---|---|
| gerçek **$0–5** | **91.8%** (tam $0: %98) | FLOOR $0 + dar band; alt-kaçış %6.8 sadece $0–2 gerçekte |
| gerçek **$20–60** | **68–72%** | ← **zayıf nokta.** Geçiş bölgesi; model hâlâ yukarı biaslı (P10 $20–40'a oturuyor), alt-kaçış %21–23. Kalıntı çöküş/toparlanma biası (+$1.54) bandda görünüyor |
| gerçek **$60–100** | 83.9% | hakim rejim, sağlam; üst-kaçış %11.8 (küçük spike'lar) |
| gerçek **$100+** | 48% (n=25) | spike'lar yukarı kaçıyor — bilinen sınır, recency kör |
| **saat** | 80–81% her grupta | saat-bazlı çözünürlük çalışıyor, düz |
| **rejim** | 80–81% her rejimde | agrega hedefte |

Yani genel %80 gerçek; ama **$20–60 gerçekleşen aralığında ~%70'e düşüyor** — modelin
sistematik yukarı biasının konformal asimetrik şiftle tam yutulamayan kısmı. C.5.5
(rejim-tetikli ağırlık, biası +$1.54 → ~+$0.5) bu cebi de daraltır.

## 8. Statik ağırlık — `base 0.5×` vs eşit ensemble

C.5.5 (rejim-tetikli ağırlık) sorusu: çöküşte parlayan bir üye yok — kısa pencereler base'ten
sadece ~\$0.8 MAE iyi, onu da toparlanmada −\$2 bias olarak geri ödüyorlar. Çöküş MAE base'in
ağırlığından bağımsız ~\$9.6'da sabit; ağırlık sadece **bias kolu**.

Dedektör riski almadan: **statik** `P50 = (0.5·base + roll90 + roll150) / 2.5`. Eşit ensemble
(`(base+roll90+roll150)/3`) ile yarıştır — rejim metrikleri, hareketli WAPE, sıfır-saat, konformal band.

In [12]:
ens_eq   = (W["base"] + W[90] + W[150]) / 3
ens_half = (0.5 * W["base"] + W[90] + W[150]) / 2.5
liveDB = dblock(pd.read_sql(text("SELECT target_ts ts, predicted_mcp_usd v FROM gold.ptf_predictions_daily "
                                 "WHERE model_name='LightGBM_v1'"), get_db_engine()).assign(
                                 ts=lambda d: pd.to_datetime(d.ts)).set_index("ts").v)

SEG8 = {"TÜM 757g": ("2024-08-01", "2026-08-27"), "normal": ("2024-08-01", "2026-01-31"),
        "çöküş": ("2026-02-01", "2026-06-30"), "çöküş-derin": ("2026-03-01", "2026-05-31"),
        "toparlanma": ("2026-07-01", "2026-08-27")}
cmp = {"canlı DB": liveDB, "eşit ensemble": ens_eq, "base 0.5× ensemble": ens_half}
o = {}
for seg, (a, b) in SEG8.items():
    for nm, d in cmp.items():
        m = M(d, a, b)
        o[(seg, nm)] = {"MAE": m["MAE"], "BIAS": m["BIAS"], "WAPE": m["WAPE"],
                        "rMAE": m["rMAE"], "sıfır_MAE": m["sifir_MAE"]}
print("=== REJİM ===")
print(pd.DataFrame(o).T.to_string())

END = pd.Timestamp("2026-08-27")
rows = []
for mo in [1, 3, 6, 12, 24]:
    a = END - pd.DateOffset(months=mo) + pd.Timedelta(days=1)
    e, h = M(ens_eq, a, END), M(ens_half, a, END)
    rows.append({"pencere": f"{mo} ay", "gün": e["gun"],
                 "eşit WAPE": e["WAPE"], "eşit MAE": e["MAE"], "eşit BIAS": e["BIAS"],
                 "0.5× WAPE": h["WAPE"], "0.5× MAE": h["MAE"], "0.5× BIAS": h["BIAS"],
                 "ΔMAE": round(h["MAE"] - e["MAE"], 2)})
print("\n=== HAREKETLİ WAPE ===")
print(pd.DataFrame(rows).set_index("pencere").to_string())

print("\n=== KONFORMAL BAND (N=60, w=0.5, FLOOR=$0) ===")
for nm, d in [("eşit ensemble", ens_eq), ("base 0.5×", ens_half)]:
    Lh, Uh = bands(d, 60, 0.5)
    r = "  ".join(f"{s}:{cw(Lh, Uh, a, b)[0]:.0f}%/${cw(Lh, Uh, a, b)[1]:.0f}"
                  for s, (a, b) in SEG7.items())
    print(f"  {nm:16s} {r}")

=== REJİM ===
                                  MAE  BIAS  WAPE   rMAE  sıfır_MAE
TÜM 757g    canlı DB             7.27  1.51  12.2  0.645       4.83
            eşit ensemble        7.04  0.78  11.8  0.628       5.22
            base 0.5× ensemble   7.09  0.67  11.9  0.633       5.30
normal      canlı DB             6.23  0.89   9.2  0.646       9.84
            eşit ensemble        6.21  0.74   9.2  0.646      12.18
            base 0.5× ensemble   6.27  0.73   9.2  0.653      12.53
çöküş       canlı DB            10.54  3.76  36.4  0.659       3.75
            eşit ensemble        9.62  1.54  33.5  0.609       3.81
            base 0.5× ensemble   9.58  1.16  33.4  0.606       3.82
çöküş-derin canlı DB            10.20  3.84  42.9  0.673       3.29
            eşit ensemble        9.20  1.80  39.1  0.620       3.37
            base 0.5× ensemble   9.16  1.47  38.9  0.617       3.39
toparlanma  canlı DB             8.63  1.51  14.4  0.602        NaN
            eşit ensemble        8

  eşit ensemble    TÜM:81%/$23  normal:81%/$21  çöküş:80%/$27  toparlanma:81%/$29  son 90g:80%/$28  son 30g:86%/$29


  base 0.5×        TÜM:81%/$23  normal:81%/$21  çöküş:80%/$27  toparlanma:80%/$30  son 90g:80%/$28  son 30g:85%/$29


**`base 0.5×` okuması** (yukarıdaki çıktı):

| dilim | eşit ensemble | base 0.5× | fark |
|---|---|---|---|
| TÜM 757g MAE / BIAS / rMAE | 7.04 / +0.78 / 0.628 | 7.09 / +0.67 / 0.633 | MAE +\$0.05, BIAS −\$0.11 |
| normal MAE / rMAE / sıfır-saat | 6.21 / 0.646 / 12.2 | 6.27 / 0.653 / 12.5 | küçük regresyon |
| **çöküş MAE / BIAS** | 9.62 / +1.54 | **9.58 / +1.16** | MAE −\$0.04, **BIAS −\$0.38** |
| çöküş-derin MAE / BIAS | 9.20 / +1.80 | 9.16 / +1.47 | aynı yön |
| toparlanma MAE / BIAS | 8.20 / −0.81 | 8.36 / −1.23 | MAE +\$0.16, bias daha negatif |
| konformal band kapsama | her rejim %80–81 | her rejim %80–81 | **değişmiyor** |
| hareketli WAPE (1–24 ay) | — | +0.02 … +0.12 pt | her ufukta minik kötü |

**Net:** `base 0.5×` çöküş bias'ında ~\$0.4 kazandırıyor, karşılığında normal + toparlanmada
~\$0.05–0.16 MAE veriyor. Konformal band (\$20–60 cebi dahil) kayıtsız — bias düzeltmesi bandın
kendi hata geçmişine de giriyor. Çöküş MAE zaten base ağırlığından bağımsız (~\$9.6 sabit);
kazanç sadece bias.

**Karar:** eşit ensemble default kalır. `base 0.5×` = elimizde duran, çöküş uzarsa tek satırla
çevrilecek statik kol. Rejim dedektörü (C.5.5) bu ~\$0.4 için kurulmaz.

**DM testi (§4.1) ek kanıt:** `base 0.5×` çöküşte eşit ensemble'dan DM-ayırt edilemez (p=0.90), normal + toparlanmada DM-daha kötü (p<0.001). Yani statik kol olarak bile ölçülebilir bir kazancı yok.

## Sonuç

**Tüm sayılar `_v2` — proxy'siz, kaskad bug'ı düzeltildikten sonra.** Framework canlıyı sadık
üretiyor: proxy'siz `base` MAE \$7.23 ≈ canlı DB \$7.27 (çöküş \$10.49 ≈ \$10.54).

| model | TÜM 757g MAE / rMAE / BIAS | normal MAE | çöküş MAE / BIAS | toparlanma MAE / BIAS |
|---|---|---|---|---|
| **canlı DB** | 7.27 / 0.645 / +1.51 | 6.23 | 10.54 / +3.76 | 8.63 / +1.51 |
| **base** (=canlı politika, proxy'siz) | 7.23 / 0.645 / +1.33 | 6.20 | 10.49 / +3.44 | 8.48 / +1.27 |
| roll90 | 7.35 / 0.656 / +0.43 | 6.51 | 9.78 / +0.51 | 9.01 / −2.13 |
| roll150 | 7.27 / 0.649 / +0.57 | 6.46 | 9.71 / +0.66 | 8.71 / −1.57 |
| **`base+roll90+roll150`** | **7.04 / 0.628 / +0.78** | **6.21** | **9.62 / +1.54** | **8.20 / −0.81** |

**Kazanan: `base + roll90 + roll150` eşit-ağırlık ensemble.**

1. **C.5.2 (özellik):** çöküş MAE −\$0.18 (bug'lı ilk turda "ölü" görünmüştü — donmuş girdi).
   Küçük gerçek etki, ama recency'nin eki, ikamesi değil.
2. **C.5.3 tek yuvarlanan pencere** çöküşü çözüyor (roll90/150 çöküş MAE ~\$9.7, BIAS ~0) ama
   **normal rejimi bozuyor** (roll90 normal MAE 6.20→6.51, sıfır-saat 10.7→14.5) ve **toparlanmada
   bias'ı +\$1.3 → −\$2.1'e** çeviriyor (yükselişi eksik tahmin). Tek pencere = rejim-geçiş gecikmesi.
3. **Çok-pencereli ensemble** LEAR ensemble mantığının LightGBM karşılığı: uzun üye (base)
   yapı+mevsimsellik+sıfır-saat, kısa üyeler (90/150) güncel rejim. Sonuç:
   - **TÜM:** canlıya karşı MAE −\$0.23 (−%3), rMAE 0.645 → 0.628
   - **çöküş:** −\$0.92 (−%9), **BIAS +\$3.76 → +\$1.54 (−%59)**
   - **toparlanma:** −\$0.43, BIAS +\$1.51 → −\$0.81 (|bias| yarıya)
   - **normal:** nötr (6.23 → 6.21). Tek maliyet: sıfır-saat MAE 9.8 → 12.2 (kısa pencereler
     normal gece saatlerini kötü tahmin ediyor, düşük fiyat-etkisi).
   - **Rejim cezası yok** — v1'in "normal sıfır-saati bozuyor, toparlanma belirsiz" resmi kaskad
     bug'ından kaynaklanıyormuş; v2'de temiz.
4. **Kalan +\$1.54 çöküş biası** tam bitmiyor. C.5.5 (rejim-tetikli ağırlık, çöküşte `base`
   üyesini kıs) → ~+\$0.5. Ani tek-gün aşağı-spike'lar (30 Ağu) hâlâ kör — tüm recency yöntemleri.
5. **C.5.4 (naive-2 router) rafta** — LEAR sızıntısı (Exogenous 2 = o gün gerçekleşen KGÜP) +
   naive-2 zayıf sinyal + rejim kapısı gerekiyor. Ensemble'dan karmaşık, daha az kazanç.

**Karar kapısı (Faz 0 adım 5):** çöküş temiz kazanç MAE −\$0.92 (eşik −\$0.88 ✓),
BIAS +\$3.76 → +\$1.54 ✓, toparlanma iyileşiyor, normal nötr. **GEÇİLDİ.**

### Canlıya taşınacak

`predict_daily_pipeline.py`: P50 = **3 model** (son 90g / 150g / tüm-geçmiş 2023-01), aynı
`lgb_lag0_v2` konfigü, P50'lerin ortalaması. Eğitim ~2× duvar-saati. **P10/P90 → §7** (konformal
band, quantile head'leri kaldır). Detaylı geçiş: `ENSEMBLE_CANLI_GECIS.md`.

### Sıradaki (plan)

- **C.5.5:** rejim-tetikli ensemble ağırlığı — çöküş sezilince `base` üyesini kıs (BIAS +\$1.54 → ~+\$0.5).
- **Faz D:** uyarlanmış LEAR (Exogenous 2 = proxy ile dürüst) vs bu ensemble.
